# 09 – HAR-12 com uma referência Suomi NPP isolada (global)

Corra primeiro o notebook 01 com `SENSORS = {'modis', 'viirs-snpp'}` para os anos necessários. A descoberta de ZIPs do notebook 01 já suporta `raw/VIIRS SNPP/viirs-snpp_<ano>_all_countries.zip`. Este notebook só lê os ficheiros processados resultantes.

Constrói séries mensais **globais** (`cell = -1`: a soma das células-dia de todas as células, `build_sensor_monthly`) independentes para MODIS e S-NPP e uma calibração S-NPP separada. **Não** lê nem substitui as tabelas L1/L6 do NOAA-20 (`viirs-jpss1`), e fica fora do notebook 02 até haver uma política explícita de seleção de satélite.

A emenda configurada usa MODIS antes de janeiro de 2012 e S-NPP a partir de janeiro de 2012. Por defeito ajusta cinco anos civis de cada lado, com efeitos de mês, tendências lineares e uma mudança de nível. A calibração usa só meses comparáveis MODIS/S-NPP em 2018–2021 e escala o segmento MODIS anterior à transição para as unidades do S-NPP. Os anos em falta continuam a ser lacunas.

Os erros-padrão são os do modelo OLS convencional e não consideram a correlação serial dos resíduos. O resultado descreve a emenda configurada e não estabelece um efeito causal do sensor. Valide os diagnósticos do modelo e os dados reais antes de interpretar as estimativas.

- **Retoma:** o resultado de cada ficheiro (modis/viirs-snpp, um ano) fica em `metrics/_partial_har12_snpp/` com assinatura (grelha, ficheiro fonte, código); `FORCE = True` refaz.
- **Saídas:** `l7_har12_snpp_monthly` e `l7_har12_snpp_factor` (as entradas que o notebook 08 relê, tabelas globais pequenas) e `l7_har12_residual_step`. A linha `har12_snpp` de `_runs` é o **sentinela** dessas entradas: é apagada antes de escrever as tabelas e escrita **por último** (com `model_options`, `missing_years` e as fontes); sem ela o notebook 08 ignora as entradas S-NPP.
- A captura de ecrã não mostrava arquivos de 2016 nem 2019: a descoberta indica os anos processados em falta sem os substituir por atividade nula.

In [ ]:
from google.colab import drive, userdata
from pathlib import Path
import base64
import os
import subprocess
import sys

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/Sprinklers')        # ajustar se o atalho tiver outro nome
PROCESSED = ROOT / 'processed'
METRICS = ROOT / 'metrics'
REPO = Path('/content/repo')
environment = os.environ.copy()
try:
    token = userdata.get('GITHUB_TOKEN')
except userdata.SecretNotFoundError:
    token = None
if token:
    credential = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
    environment.update(GIT_CONFIG_COUNT='1',
                       GIT_CONFIG_KEY_0='http.https://github.com/.extraheader',
                       GIT_CONFIG_VALUE_0=f'AUTHORIZATION: basic {credential}')
    del credential
if (REPO / '.git').exists():
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin'], env=environment, check=True)
    subprocess.run(['git', '-C', str(REPO), 'switch', 'metricas-pipeline'], env=environment, check=True)
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', 'metricas-pipeline'],
                   env=environment, check=True)
else:
    subprocess.run(['git', 'clone', '--branch', 'metricas-pipeline',
                    'https://github.com/Rui-Martins31/NASA_Space_Apps_2026_Sprinklers.git',
                    str(REPO)], env=environment, check=True)
del token, environment
sys.path.insert(0, str(REPO / 'data' / 'pipeline'))

assert PROCESSED.exists(), f"Nao encontro {PROCESSED}. Corra primeiro o notebook 01 e confirme o atalho 'Sprinklers'."
METRICS.mkdir(exist_ok=True)


In [ ]:
# cell: params
FORCE = False
GRID_DEG = 0.01                      # grelha das celulas-dia (igual a dos notebooks 02/07/08)
CAL_START, CAL_END, TEST_START = 2018, 2021, 2022
MIN_CAL_MONTHS = 24
MODEL_OPTIONS = dict(reference_sensor='viirs-snpp', transition_year=2012, window_years=5,
                     min_months_each_side=36, min_years_each_side=3, max_unknown_days=0)
SNPP_LOTE = 'har12_snpp'             # linha de `_runs` que serve de sentinela das entradas HAR-12


In [ ]:
# cell: defs
import datetime, hashlib, importlib, json, re, subprocess, tempfile
from pathlib import Path
import duckdb
import numpy as np
import pandas as pd

from metrics import common, harmonization_snpp, harmonization_transition
from metrics.harmonization_factor import har06_factor
from metrics.store import ParquetSink, write_run, write_table

for module in (harmonization_snpp, harmonization_transition):
    importlib.reload(module)

common._grid_shape(GRID_DEG)
SINK = ParquetSink(METRICS)
PART = METRICS / '_partial_har12_snpp'
PART.mkdir(exist_ok=True)
COLUMNS = ['instrument', 'satellite', 'latitude', 'longitude', 'acq_date', 'type', 'confidence_class']
FILE_RE = re.compile(r'(modis|viirs-snpp)_([0-9]{4})')


def atomic_part(df, path):
    """Grava um parcial Parquet (colunas de texto sempre como `string`, mesmo sem valores)."""
    out = df.copy()
    for column in out.columns:
        if pd.api.types.is_object_dtype(out[column].dtype) or pd.api.types.is_string_dtype(out[column].dtype):
            out[column] = out[column].astype('string')
    tmp = path.with_name(path.name + '.tmp')
    out.to_parquet(tmp, index=False)
    tmp.replace(path)


def atomic_json(value, path):
    tmp = path.with_name(path.name + '.tmp')
    tmp.write_text(json.dumps(value, sort_keys=True), encoding='utf-8')
    tmp.replace(path)


def read_json(path):
    try:
        return json.loads(path.read_text(encoding='utf-8'))
    except (OSError, ValueError):
        return None


def file_digest(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def tree_digest(name):
    """Assinatura de uma tabela publicada (pasta Parquet): ficheiros, tamanhos e mtimes."""
    root = SINK.path(name)
    items = sorted((p.relative_to(root).as_posix(), p.stat().st_size, p.stat().st_mtime_ns)
                   for p in root.rglob('*.parquet'))
    return hashlib.sha256(json.dumps(items).encode()).hexdigest()


def frame_digest(df):
    values = pd.util.hash_pandas_object(df.reset_index(drop=True), index=False).to_numpy().tobytes()
    return hashlib.sha256(values + ','.join(map(str, df.columns)).encode()).hexdigest()


def duckdb_connect():
    con = duckdb.connect()
    con.execute(f"SET temp_directory='{Path(tempfile.gettempdir()).as_posix()}/duckdb_spill'")
    return con


def sql_path(path):
    return str(path).replace('\\', '/').replace("'", "''")


def sql_list(files):
    return '[' + ', '.join("'" + sql_path(f) + "'" for f in files) + ']'


def table_source(name):
    """Fonte DuckDB que le a pasta Parquet publicada `name` (com as particoes hive)."""
    pattern = sql_path(SINK.path(name) / '**' / '*.parquet')
    return f"read_parquet('{pattern}', hive_partitioning=true, union_by_name=true)"


def scalar(name, expression):
    """Valor de uma agregacao SQL sobre a tabela publicada `name`."""
    con = duckdb_connect()
    try:
        value = con.execute(f'SELECT {expression} FROM {table_source(name)}').fetchone()[0]
    finally:
        con.close()
    return 0 if value is None else value


def consolidate(name, files, order_by, keys):
    """Junta os parciais `files` na tabela publicada `name` (um ficheiro Parquet ordenado por `order_by`,
    com DuckDB fora da memoria). Verifica chaves repetidas e devolve o numero de linhas.
    Nao se particiona por `lat_band`: estas tabelas levam linhas globais (cell = -1, sem lat_band)."""
    files = [Path(f) for f in files]
    source = f'read_parquet({sql_list(files)}, union_by_name=true)'
    con = duckdb_connect()
    try:
        n_rows = con.execute(f'SELECT count(*) FROM {source}').fetchone()[0]
        if n_rows:
            repeated = con.execute(
                f'SELECT count(*) FROM (SELECT 1 FROM {source} GROUP BY {", ".join(keys)} HAVING count(*) > 1)'
            ).fetchone()[0]
            assert repeated == 0, f'{name}: {repeated} chaves repetidas entre parciais'
    finally:
        con.close()
    if n_rows == 0:
        SINK.write(name, pd.read_parquet(files[0]).iloc[:0], ())
        return 0
    SINK.write_from_parquet(name, files, order_by=order_by, partition_cols=())
    return int(n_rows)


def plain(df):
    """Colunas de particao (hive) voltam como categoricas: passa-as a tipos normais."""
    for column in df.columns:
        if isinstance(df[column].dtype, pd.CategoricalDtype):
            df[column] = df[column].astype(df[column].cat.categories.dtype)
    return df


def git_commit():
    try:
        return subprocess.run(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], capture_output=True,
                              text=True).stdout.strip() or 'unknown'
    except Exception:
        return 'unknown'


def drop_run(lote):
    """Apaga a linha `lote` de `_runs` (invalida um sentinela anterior antes de reescrever as entradas)."""
    if SINK.exists('_runs'):
        runs = SINK.read('_runs')
        SINK.write('_runs', runs[runs['lote'] != lote].reset_index(drop=True), ())


In [ ]:
# cell: build
COMMIT = git_commit()
transition_year = MODEL_OPTIONS['transition_year']
calibration_years = set(range(CAL_START, CAL_END + 1))
required_years = {
    'modis': set(range(transition_year - MODEL_OPTIONS['window_years'], transition_year)) | calibration_years,
    'viirs-snpp': set(range(transition_year, transition_year + MODEL_OPTIONS['window_years'])) | calibration_years,
}
sources = []
for path in sorted(PROCESSED.glob('*.parquet')):
    match = FILE_RE.fullmatch(path.stem)
    if match and int(match[2]) in required_years[match[1]]:
        sources.append((match[1], int(match[2]), path))
for sensor in ['modis', 'viirs-snpp']:
    if not any(source[0] == sensor for source in sources):
        raise FileNotFoundError(f'Missing processed {sensor} files in {PROCESSED}. Run notebook 01 first.')
missing_years = {}
for sensor in ['modis', 'viirs-snpp']:
    present = {year for selected_sensor, year, _ in sources if selected_sensor == sensor}
    missing_years[sensor] = sorted(required_years[sensor] - present)
print('Missing processed years, retained as gaps:', missing_years)

code_digest = file_digest(harmonization_snpp.__file__)
parts = []
for sensor, year, path in sources:
    part = PART / f'{sensor}_{year}.parquet'
    marker = PART / f'{sensor}_{year}.json'
    signature = dict(schema=1, sensor=sensor, year=year, grid_deg=GRID_DEG, code=code_digest,
                     source=dict(name=path.name, size=path.stat().st_size, mtime_ns=path.stat().st_mtime_ns))
    if not FORCE and part.is_file() and read_json(marker) == signature:
        parts.append(pd.read_parquet(part))
        print(f'Reusing global monthly series for {sensor} {year}')
        continue
    marker.unlink(missing_ok=True)
    frame = pd.read_parquet(path, columns=COLUMNS)
    dates = pd.to_datetime(frame.acq_date, errors='coerce', utc=True).dropna()
    if not dates.dt.year.eq(year).all():
        raise ValueError(f'Acquisition dates outside filename year in {path}')
    monthly_part = harmonization_snpp.build_sensor_monthly(frame, sensor=sensor, grid_deg=GRID_DEG)
    del frame
    atomic_part(monthly_part, part)
    atomic_json(signature, marker)
    parts.append(monthly_part)
    print(f'Processed global monthly series for {sensor} {year}')
source_monthly = pd.concat(parts, ignore_index=True)
monthly = source_monthly.drop(columns='sensor')
factors = har06_factor(monthly, min_cal_months=MIN_CAL_MONTHS, cal_start=CAL_START, cal_end=CAL_END,
                       test_start=TEST_START, max_unknown_days=MODEL_OPTIONS['max_unknown_days'],
                       include_global=False)
result = harmonization_transition.har12_residual_step(monthly, factors, **MODEL_OPTIONS)


In [ ]:
# cell: publish
drop_run(SNPP_LOTE)      # invalida o sentinela: uma exportacao interrompida nao deixa entradas validas
published = {}
for name, table in [('l7_har12_snpp_monthly', monthly), ('l7_har12_snpp_factor', factors),
                    ('l7_har12_residual_step', result)]:
    write_table(SINK, name, table, grid_deg=GRID_DEG, partition_by=[])     # globais pequenas: um ficheiro, sem particoes
    published[name] = len(table)
# `_runs` escreve-se POR ULTIMO: e o sentinela que o notebook 08 le (model_options incluidos)
write_run(SINK, SNPP_LOTE, commit=COMMIT, grid_deg=GRID_DEG, tables=published, params=dict(
    model_options=MODEL_OPTIONS, grid_deg=GRID_DEG, cal_start=CAL_START, cal_end=CAL_END, test_start=TEST_START,
    min_cal_months=MIN_CAL_MONTHS, missing_years=missing_years,
    sources=[dict(sensor=sensor, year=year, name=path.name, size=path.stat().st_size, mtime_ns=path.stat().st_mtime_ns)
             for sensor, year, path in sources],
    run_utc=datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds')))
print(result.to_string(index=False))
print('HAR-12 S-NPP outputs written to', METRICS)


In [ ]:
# cell: verify
assert (monthly['cell'] == common.GLOBAL_CELL).all(), 'a serie mensal de HAR-12 tem de ser global'
assert not monthly.duplicated(['year', 'month', 'series']).any()
assert set(monthly['series']) <= {'modis', 'viirs'}
assert len(result) <= 1 and (result['cell'] == common.GLOBAL_CELL).all()
runs = SINK.read('_runs')
assert runs.loc[runs.lote.eq(SNPP_LOTE)].shape[0] == 1, 'falta o sentinela har12_snpp em _runs'
for name, rows in published.items():
    assert len(SINK.read(name)) == rows, f'{name}: linhas gravadas != linhas calculadas'
print('OK: entradas HAR-12 S-NPP verificadas')
